# BÁO CÁO GIỮA KỲ: Đề xuất và cài đặt phiên bản tuần tự

**Khoa:** Công nghệ thông tin

**Trường:** Đại học Khoa học tự nhiên, ĐHQG-HCM

**GVHD:** thầy Phạm Trọng Nghĩa

**Nhóm:** 07

**Năm học:** 2026 – 2027

---

## 1. Thông tin nhóm và kế hoạch công việc

### 1.1 Danh sách thành viên

| STT | MSSV      | Họ và tên             | Vai trò |
|:---:|-----------|-----------------------|---------|
| 1   | 23120315  | Nguyễn Đăng Pha       |         |
| 2   | 23120356  | Trần Nguyễn Trí Thanh |         |
| 3   | 23120360  | Đặng Lê Đức Thịnh     |         |
| 4   | 23120366  | Trương Thanh Thuận    |         |

### 1.2 Kế hoạch phân công

| Giai đoạn - Công việc                                       | Thành viên phụ trách   | Phiên bản  |
|-------------------------------------------------------------|------------------------|------------|
| Giai đoạn 1 - Tiền xử lý (Preprocessing)                    |                        | V0         |
| Giai đoạn 2 - Phát hiện điểm đặc trưng (Keypoint Detection) |                        | V0         |
| Giai đoạn 3 - Mô tả điểm đặc trưng (Description)            |                        | V0         |
| Giai đoạn 4 - Ghép cặp điểm (Matching)                      |                        | V0         |
| Giai đoạn 5 - Ước lượng Homography                          |                        | V0         |
| Giai đoạn 6 - Biến dạng và hòa trộn (Warping & Blending)    |                        | V0         |
| Đánh giá hiệu năng và báo cáo                               |                        | Tất cả     |

---

## 2. Hiểu vấn đề

### 2.1 Bài toán ghép ảnh toàn cảnh (Panorama Stitching)

Ghép ảnh toàn cảnh là quá trình kết hợp hai (hoặc nhiều) ảnh chụp có vùng chồng lấp của cùng một cảnh vật thành một ảnh góc rộng duy nhất...

**Đầu vào:** Hai ảnh RGB có vùng chồng lấp (ảnh A và ảnh B).  
**Đầu ra:** Một ảnh toàn cảnh duy nhất (ảnh B được biến dạng và hòa vào khung của ảnh A).

### 2.2 Pipeline 6 giai đoạn

| #   | Giai đoạn                    | Mô tả                                                                         |
|:---:|------------------------------|-------------------------------------------------------------------------------|
| 1   | **Tiền xử lý**               | Chuyển ảnh sang ảnh xám (grayscale) và làm mịn để giảm nhiễu                  |
| 2   | **Phát hiện điểm đặc trưng** | Tìm các điểm nổi bật (góc cạnh) trong từng ảnh                                |
| 3   | **Mô tả điểm đặc trưng**     | Tính vector mô tả nhỏ gọn cho vùng lân cận mỗi điểm                           |
| 4   | **Ghép cặp điểm**            | Với mỗi điểm ở ảnh A, tìm điểm tương đồng nhất ở ảnh B; loại bỏ các cặp mơ hồ |
| 5   | **Ước lượng Homography**     | Tính ma trận biến đổi xạ ảnh 3×3 để ánh xạ ảnh B sang ảnh A, xử lý nhiễu      |
| 6   | **Biến dạng và hòa trộn**    | Biến dạng ảnh B vào hệ tọa độ của ảnh A và hòa trộn vùng chồng lấp            |

---

## 3. Phương pháp được chọn

Nhóm chọn phương pháp cho từng giai đoạn dựa trên hai tiêu chí: **chất lượng đầu ra** và **khả năng tăng tốc trên GPU**.

<table>
  <thead>
    <tr>
      <th>Giai đoạn</th>
      <th>Phương pháp lựa chọn</th>
      <th>Cơ chế hoạt động</th>
      <th>Tham số chính</th>
      <th>Ưu thế chất lượng</th>
      <th>Tính phù hợp với GPU</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1. Tiền xử lý</td>
      <td>Grayscale cố định + Separable Gaussian Blur (1D Horiz → 1D Vert)</td>
      <td>Chuyển RGB→Gray bằng công thức BT.601; nhân chập 1D theo hàng rồi theo cột thay vì kernel 2D đầy đủ</td>
      <td>σ (độ lệch chuẩn), k = 2⌈3σ⌉+1</td>
      <td>Loại bỏ nhiễu tần cao hiệu quả; O(2kN) thay vì O(k²N) mỗi pixel</td>
      <td>Mỗi pixel độc lập, hai pass 1D dễ map 1 thread/pixel; cache-friendly hơn kernel 2D</td>
    </tr>
    <tr>
      <td rowspan="2">2. Phát hiện điểm đặc trưng</td>
      <td>FAST-9/12 + Grid-based Top-K</td>
      <td>Kiểm tra vòng tròn 16 pixel quanh tâm; điểm là góc nếu >= N pixel liên tiếp sáng/tối hơn ngưỡng t; lọc Top-K theo score trên mỗi ô lưới</td>
      <td>t (intensity threshold), N trong {9, 12}, kích thước ô lưới, K (số điểm/ô)</td>
      <td>Cực nhanh, không cần tích chập nặng; Grid Top-K đảm bảo phân bố đều keypoint</td>
      <td>16 pixel/tâm độc lập hoàn toàn; 1 thread/pixel; ô lưới phân vùng tự nhiên cho CUDA block</td>
    </tr>
    <tr>
      <td>Harris Corner + NMS Top-K</td>
      <td>Tính gradient Ix/Iy (Sobel 3×3); xây dựng structure tensor M; tính Harris response R = det(M) − k·trace(M)²; giữ điểm R > ngưỡng và áp NMS</td>
      <td>k (Harris sensitivity ~0.04–0.06), σ_w (Gaussian weight tensor), R_thresh, NMS radius</td>
      <td>Ổn định hơn với rotation và viewpoint change; response liên tục cho phép subpixel</td>
      <td>Mỗi bước (Sobel, tensor, response) là phép tính pixel-wise; NMS song song theo vùng</td>
    </tr>
    <tr>
      <td>3. Mô tả điểm đặc trưng</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>4. Ghép cặp điểm</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>5. Ước lượng Homography</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>6. Biến dạng và hòa trộn</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
  </tbody>
</table>


### 3.1 Giai đoạn 1 – Tiền xử lý: Grayscale cố định + Separable Gaussian Blur

#### Cơ chế hoạt động

**Bước 1 – RGB → Grayscale (công thức cố định BT.601):**

```
Gray = 0.299·R + 0.587·G + 0.114·B
```

Trọng số cố định → mỗi pixel RGB được xử lý **độc lập hoàn toàn**, không có sự phụ thuộc dữ liệu giữa các pixel.

**Bước 2 – Separable Gaussian Blur (tách kernel 2D thành 2 lần nhân chập 1D):**

Tận dụng tính chất phân ly (separability) của kernel Gaussian:

```
G(x,y) = g(x) · g(y)  →  Pass 1: nhân chập 1D theo hàng
                         Pass 2: nhân chập 1D theo cột
```

Độ phức tạp giảm từ **O(k²·N)** xuống **O(2k·N)** với N là tổng số pixel và k là kích thước kernel.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| σ | Độ lệch chuẩn Gaussian, kiểm soát mức độ làm mịn | 1.0 – 1.6 |
| k | Kích thước kernel, k = 2⌈3σ⌉+1 | 7 (với σ = 1.0) |

#### Lý do phù hợp với song song hóa

- **Grayscale:** Mỗi pixel độc lập → map trực tiếp `1 thread = 1 pixel`, không cần đồng bộ.
- **Pass 1D ngang:** Mỗi hàng (row) độc lập → song song hóa theo hàng.
- **Pass 1D dọc:** Mỗi cột (column) độc lập → tương tự.
- **Cache-friendly:** Đọc bộ nhớ tuần tự 1D thay vì nhảy cách 2D → giảm cache miss đáng kể.


### 3.2 Giai đoạn 2 – Phát hiện điểm đặc trưng: FAST-9/12 vs Harris Corner

Nhóm triển khai cả hai phương pháp trong phiên bản V0 để có cơ sở đánh giá thực nghiệm, từ đó chọn phương pháp chính cho V1/V2.

---

#### Phương pháp 2a – FAST-9/12 + Grid-based Top-K

**Cơ chế:**

Với mỗi pixel trung tâm `p`, xét vòng tròn Bresenham bán kính 3 gồm **16 pixel cố định** (p1...p16).  
`p` là góc nếu tồn tại N pixel liên tiếp (N = 9 hoặc 12) thỏa:

```
I(pi) > I(p) + t   hoặc   I(pi) < I(p) - t
Score = Tổng |I(pi) - I(p)|  (dùng để xếp hạng)
```

**Grid-based Top-K:** Chia ảnh thành lưới M x N ô; mỗi ô giữ K điểm có score cao nhất để phân bố đều keypoint.

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| t | Intensity threshold | 20 – 40 |
| N | Số pixel liên tiếp (FAST-9 hoặc FAST-12) | 9 |
| Grid size | Kích thước ô lưới (px) | 32 x 32 |
| K | Keypoint giữ lại mỗi ô | 5 – 10 |

**Ưu điểm:**
- Chỉ dùng phép so sánh cường độ, không yêu cầu tích chập hay đạo hàm
- Mỗi pixel xử lý hoàn toàn độc lập, không phụ thuộc kết quả lân cận
- 16 offset vòng tròn Bresenham là hằng số, tính trước một lần và tái sử dụng
- Grid Top-K kiểm soát phân bố đều keypoint trên toàn ảnh

**Nhược điểm:**
- Không mã hóa thông tin hướng (orientation) của vùng lân cận, descriptor có thể kém ổn định khi ảnh bị xoay
- Nhạy cảm với nhiễu ảnh nếu ngưỡng t được chọn quá thấp, dễ sinh nhiều false positive
- Score chỉ phản ánh tổng sai lệch cường độ, không mô tả hình dạng hay hướng cấu trúc cục bộ

---

#### Phương pháp 2b – Harris Corner + NMS Top-K

**Cơ chế:**

```
Bước 1: Tính gradient  Ix = Sobel-X(I),  Iy = Sobel-Y(I)
Bước 2: Structure tensor  M = [[Ix^2, Ix*Iy], [Ix*Iy, Iy^2]]  (Gaussian-weighted)
Bước 3: Harris response  R = det(M) - k * trace(M)^2
Bước 4: Giữ điểm có R > threshold, áp NMS (hoặc Grid Top-K)
```

Tensor M mô tả hướng và độ lớn của biến đổi cường độ trong vùng lân cận mỗi pixel. Khi cả hai eigenvalue của M đều lớn, pixel đó nằm ở vùng góc. Harris response R là xấp xỉ nhanh của điều kiện này mà không cần tính eigenvalue trực tiếp.

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| k | Harris sensitivity (Harris-Stephens) | 0.04 – 0.06 |
| sigma_w | Gaussian weight cho structure tensor | 1.0 – 1.5 |
| R_thresh | Ngưỡng response | tự động = max(R) x 0.01 |
| NMS radius | Bán kính loại trừ điểm lân cận | 5 – 10 px |

**Ưu điểm:**
- Response dựa trên structure tensor phản ánh đặc trưng hình học cục bộ theo nhiều hướng
- Gaussian weighting trong tensor giúp giảm ảnh hưởng của nhiễu pixel đơn lẻ
- Mỗi bước (Sobel, tensor, response) là phép tính pixel-wise, thuận tiện cho song song hóa
- Kernel Gaussian từ Giai đoạn 1 có thể tái sử dụng cho bước weighting tensor

**Nhược điểm:**
- Pipeline nhiều bước nối tiếp đòi hỏi cấp phát buffer trung gian cho gradient và tensor
- NMS cần truy cập giá trị response của vùng lân cận, tạo data dependency nhẹ khi song song hóa
- Kém nhạy với các góc có kích thước nhỏ hơn cửa sổ Gaussian dùng cho tensor


---

## 4. Phiên bản tuần tự (V0)

Phiên bản V0 là cài đặt C/C++ **đơn luồng** (không dùng GPU) của toàn bộ pipeline ghép ảnh.

---

## 5. Kết quả và phân tích hiệu năng V0

---

### 5.1 Độ chính xác

### 5.2 Ảnh toàn cảnh đầu ra

### 5.3 Bảng thời gian thực thi theo từng giai đoạn

### 5.4 Phân tích giai đoạn chiếm nhiều thời gian nhất

### 5.5 Ước lượng theo Định luật Amdahl

---

## 6. Bộ dữ liệu

### 6.1 Quy chuẩn ảnh và cách chọn cặp ảnh

Mỗi cặp ảnh được **cắt tỉ lệ 16:9** và resize về 3 độ phân giải:

| Tên | Độ phân giải | Mục đích                          |
|-----|--------------|-----------------------------------|
| Nhỏ | 1280 × 720   | Phát triển nhanh và debug         |
| Vừa | 1920 × 1080  | Benchmark chính                   |
| Lớn | 3840 × 2160  | Kiểm tra khả năng mở rộng của GPU |

### 6.2 Danh sách địa điểm chụp

| STT | Địa điểm    | Bối cảnh      | Phân loại   | Mô tả cặp ảnh   | Đường dẫn |
|-----|-------------|---------------|-------------|-----------------|-----------|
| 1   |             | Indoor        | Standard    |                 |           |
| 2   |             | Outdoor       | Standard    |                 |           |
| 3   |             | Team's choice | Standard    |                 |           |
| 4   |             | Team's choice | Challenging |                 |           |

**Link bộ dữ liệu (Google Drive):** ...

---

## 7. Kế hoạch GPU

### 7.1 Các giai đoạn sẽ chuyển lên GPU

### 7.2 Chi tiết ánh xạ luồng V1

### 7.3 Ý tưởng tối ưu hóa cho V2, V3, ...

1. **[Ý tưởng 1]**

2. **[Ý tưởng 2]**

3. **[Ý tưởng 3]**

---

## Tài liệu tham khảo